# Notebook 01 — Setup & Data Download

This notebook walks through:
1. Verifying the project environment
2. Loading the project configuration
3. Downloading NUTS-2 boundary shapefiles
4. Downloading ERA5-Land climate data from Copernicus CDS

**Prerequisites:** Run `uv sync` in the project root before starting.

## 1. Verify Environment & Imports

In [ ]:
import sys
import os

# Ensure project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")

In [ ]:
# Test that key packages are available
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"xarray:     {xr.__version__}")
print(f"geopandas:  {gpd.__version__}")
print("\n✅ All key packages imported successfully")

## 2. Load Project Configuration

In [ ]:
from src.utils.config import load_config, get_path, get_region_mapping

cfg = load_config()

print(f"Study period: {cfg['study']['start_year']}–{cfg['study']['end_year']}")
print(f"Summer months: {cfg['study']['summer_months']}")
print(f"Country: {cfg['study']['country']}")
print(f"NUTS level: {cfg['study']['nuts_level']}")

In [ ]:
# View all 20 Italian NUTS-2 regions
regions = get_region_mapping(cfg)
print(f"Number of regions: {len(regions)}\n")
for code, name in regions.items():
    print(f"  {code}: {name}")

In [ ]:
# Check heatwave parameters
hw_cfg = cfg['heatwave']
print(f"Heatwave threshold: {hw_cfg['percentile_threshold']}th percentile")
print(f"Min consecutive days: {hw_cfg['min_consecutive_days']}")
print(f"Reference period: {hw_cfg['reference_start_year']}–{hw_cfg['reference_end_year']}")
print(f"Sensitivity thresholds to test: {hw_cfg['sensitivity_thresholds']}")

In [ ]:
# Check data directories exist
from pathlib import Path

for key in ['raw_data', 'interim_data', 'processed_data', 'figures', 'tables']:
    p = get_path(cfg, key)
    status = '✅' if p.exists() else '❌'
    print(f"  {status} {key}: {p}")

## 3. Download NUTS-2 Boundaries

We download the Eurostat GISCO NUTS-2021 boundaries (1:1M scale) for mapping
and spatial aggregation of climate data.

In [ ]:
from src.data.nuts2_boundaries import setup_boundaries, load_italy_nuts2

# Download boundaries (skips if already downloaded)
italy_nuts2 = setup_boundaries(cfg)

print(f"\nLoaded {len(italy_nuts2)} Italian NUTS-2 regions")
print(f"CRS: {italy_nuts2.crs}")
print(f"\nRegions:")
print(italy_nuts2[['NUTS_ID']].to_string())

In [ ]:
# Quick map to verify boundaries look correct
fig, ax = plt.subplots(1, 1, figsize=(8, 10))
italy_nuts2.plot(ax=ax, edgecolor='black', facecolor='lightblue', linewidth=0.5)
ax.set_title('Italian NUTS-2 Regions', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. Download ERA5-Land Climate Data

This downloads hourly 2m temperature and 2m dewpoint temperature from the
Copernicus Climate Data Store (CDS) for Italy's bounding box.

**⚠️ Prerequisites:**
- Register at https://cds.climate.copernicus.eu/
- Accept the ERA5-Land licence
- Set your API key: `export CDS_API_KEY=<your-key>`
  or add it to `config/config.yaml` under `cds_api.key`

**⏱ Note:** Each year takes ~15–30 min to download depending on your connection.
The full 11-year download may take several hours.

In [ ]:
# Check if CDS API is configured
try:
    from src.utils.config import get_cds_api_key
    key = get_cds_api_key(cfg)
    print(f"✅ CDS API key found (starts with: {key[:8]}...)")
except ValueError as e:
    print(f"❌ {e}")
    print("\nTo set up:")
    print("  1. Register at https://cds.climate.copernicus.eu/")
    print("  2. Go to your profile → API Key")
    print("  3. Run: export CDS_API_KEY=<your-key>")

In [ ]:
# Show what will be downloaded
temp_cfg = cfg['temperature']
bbox = temp_cfg['bbox']

print("ERA5-Land Download Configuration:")
print(f"  Variables: {temp_cfg['variables']}")
print(f"  Bounding box: N={bbox['north']}, S={bbox['south']}, W={bbox['west']}, E={bbox['east']}")
print(f"  Years: {cfg['study']['start_year']}–{cfg['study']['end_year']}")
print(f"  Months: {cfg['study']['summer_months']} (June–September)")
print(f"  Temporal resolution: Hourly (24 time steps/day)")
print(f"  Spatial resolution: ~9 km")
print(f"\n  Estimated files: {cfg['study']['end_year'] - cfg['study']['start_year'] + 1}")
print(f"  Estimated total size: ~5–10 GB")

In [ ]:
# Download a single year first to test (uncomment to run)

# from src.data.download_era5 import download_era5_year
#
# output_dir = get_path(cfg, 'raw_data') / 'climate'
# output_dir.mkdir(parents=True, exist_ok=True)
#
# # Test with 2022 first (most important year)
# download_era5_year(2022, cfg, output_dir)

In [ ]:
# Download all years (uncomment to run — takes several hours!)

# from src.data.download_era5 import download_all_era5
# files = download_all_era5(cfg)
# print(f"Downloaded {len(files)} files")

In [ ]:
# Check what climate files are already downloaded
climate_dir = get_path(cfg, 'raw_data') / 'climate'
nc_files = sorted(climate_dir.glob('*.nc'))

if nc_files:
    print(f"Found {len(nc_files)} ERA5-Land NetCDF files:")
    for f in nc_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name}  ({size_mb:.1f} MB)")
else:
    print("No ERA5-Land files found yet.")
    print("Run the download cell above, or use the CLI:")
    print("  uv run python scripts/download_data.py --source era5")

## 5. Check Mortality & Socioeconomic Data

These datasets must be downloaded manually (see `data_download_guide.md`).
Check what's been placed in the raw data directories.

In [ ]:
# Check mortality data directory
mort_dir = get_path(cfg, 'raw_data') / 'mortality'
mort_files = [f for f in mort_dir.iterdir() if f.name != '.gitkeep']

if mort_files:
    print(f"✅ Found {len(mort_files)} mortality files:")
    for f in mort_files:
        print(f"  {f.name}")
else:
    print("❌ No mortality data found in data/raw/mortality/")
    print("   See data_download_guide.md for download instructions.")

In [ ]:
# Check socioeconomic data directory
socio_dir = get_path(cfg, 'raw_data') / 'socioeconomic'
socio_files = [f for f in socio_dir.iterdir() if f.name != '.gitkeep']

if socio_files:
    print(f"✅ Found {len(socio_files)} socioeconomic files:")
    for f in socio_files:
        print(f"  {f.name}")
else:
    print("❌ No socioeconomic data found in data/raw/socioeconomic/")
    print("   See data_download_guide.md for download instructions.")

## Summary

| Data Source | Status | Next Step |
|---|---|---|
| NUTS-2 boundaries | ✅ Downloaded | Proceed to Notebook 02 |
| ERA5-Land climate | ⏳ Check above | Download via CDS API |
| Mortality (ISTAT) | ⏳ Check above | Manual download (see guide) |
| Socioeconomic | ⏳ Check above | Manual download (see guide) |

➡️ **Next:** `02_process_data.ipynb` — Process raw data into analysis-ready format.